# 我們平台訓練 — Forecasting Sticker Sales

時序回歸 — 預測各國 × 各店 × 各產品的 sticker 日銷量。本 notebook 把平台**完整流程**搬上來:

1. **平台預處理** (`preprocess_for_training`): RobustDataCleaner 拆 datetime,AutoRouter 自動分類欄位,雙軌 (Tree/DL) 組裝
2. **平台訓練** (`pipeline_time.run_regression`): 雙軌 ensemble + Optuna HPO + Nelder-Mead blender + Meta-learner stacker
3. **Target 自動轉換**: 右偏正值 target (銷量) 自動 log1p → 推論期 expm1 + clip≥0
4. 輸出 `/kaggle/working/submission.csv`,Kaggle 一鍵 submit

## 前置作業 (一次性)
1. 把專案的 `api/` 整個資料夾上傳成一份 Kaggle Dataset(名字隨意,自動偵測)
2. 本 notebook 的 **Settings → Add Input** → 加上這份 dataset + Sticker 比賽資料
3. (可選) Accelerator 選 GPU T4 x2 — daniel DL 軌會用上

## Step 1 ─ 環境準備

In [ ]:
import sys, os, subprocess

PLATFORM_ROOT = None
for d in os.listdir('/kaggle/input'):
    p = os.path.join('/kaggle/input', d)
    if os.path.isdir(os.path.join(p, 'api', 'train', 'pipeline')):
        PLATFORM_ROOT = p
        break
assert PLATFORM_ROOT, '找不到平台 code — 請先 attach 你上傳的 api/ 那份 Kaggle Dataset'
print(f'[platform] root = {PLATFORM_ROOT}')

sys.path.insert(0, PLATFORM_ROOT)
sys.path.insert(0, os.path.join(PLATFORM_ROOT, 'api', 'train', 'pipeline'))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn', 'shap'], check=False)
print('[deps] OK')

In [ ]:
# 比賽資料自動偵測
def find_comp_data():
    candidates = []
    for dirpath, _dirs, files in os.walk('/kaggle/input'):
        if PLATFORM_ROOT in dirpath:
            continue
        if 'train.csv' in files and 'test.csv' in files:
            size = os.path.getsize(os.path.join(dirpath, 'train.csv'))
            candidates.append((dirpath, size))
    if not candidates:
        return None
    candidates.sort(key=lambda x: (0 if 'new' in os.path.basename(x[0]).lower() else 1, -x[1]))
    return candidates[0][0]

DATA_DIR = find_comp_data()
assert DATA_DIR, '找不到比賽資料 — 請 attach Forecasting Sticker Sales'
print(f'[data] {DATA_DIR}')

def has_gpu():
    try:
        r = subprocess.run(['nvidia-smi'], capture_output=True, timeout=3)
        return r.returncode == 0
    except Exception:
        return False
print(f'[gpu]  {has_gpu()}')

## Step 2 ─ 載入 + 簡單 EDA

In [ ]:
import pandas as pd, numpy as np
import time, warnings, gc
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

TARGET = 'num_sold'
ID_COL = 'id'

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
print(f'train: {train_df.shape}  test: {test_df.shape}')
print(f'\ndtypes:\n{train_df.dtypes}')
print(f'\nNaN: target={train_df[TARGET].isna().sum()} ({train_df[TARGET].isna().mean()*100:.2f}%)')

# Target 分佈 — 看是不是右偏 (右偏才適合 log1p)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
y_valid = train_df[TARGET].dropna()
axes[0].hist(y_valid, bins=80, color='steelblue')
axes[0].set_title(f'num_sold  skew={y_valid.skew():.2f}  (越大越右偏)')
axes[0].set_xlabel('num_sold'); axes[0].set_ylabel('count')
axes[1].hist(np.log1p(y_valid), bins=80, color='coral')
axes[1].set_title(f'log1p(num_sold)  skew={np.log1p(y_valid).skew():.2f}  (接近常態才會學得準)')
axes[1].set_xlabel('log1p(num_sold)'); axes[1].set_ylabel('count')
plt.tight_layout(); plt.show()

## Step 3 ─ 平台預處理 (RobustDataCleaner + AutoRouter + PipelineAssembler)

Sticker 有 `date` (datetime 字串)、`country`/`store`/`product` (類別文字),平台預處理會:
- 自動拆 `date` → year/month/dayofweek/dayofyear
- AutoRouter 把類別文字判成 categorical → PipelineAssembler 接 OHE
- 輸出雙軌(Tree 生肉版 / DL 熟肉版),這裡用 Tree 軌餵 daniel

**先 drop NaN label**(後面 log1p 不能容忍 NaN)。

In [ ]:
# Drop NaN label rows
before = len(train_df)
train_df = train_df.dropna(subset=[TARGET]).reset_index(drop=True)
print(f'[clean] drop {before - len(train_df)} 筆 num_sold NaN — 剩 {len(train_df)}')

from api.preprocess import preprocess_for_training

# 平台預處理:test_size=0 表示 train 全用,不切 holdout(我們要用全部資料訓練 → 預測 Kaggle test)
X_train_dict, X_test_dict, y_train, y_test, fitted_preprocessors = preprocess_for_training(
    train_df, TARGET, test_size=0.0,
    is_time_series=True,        # Sticker 是時序資料,維持時間順序
)
X_train_tree = X_train_dict['tree']
tree_preprocessor = fitted_preprocessors['tree']
print(f'\n[preprocess] tree 軌道輸出: {X_train_tree.shape}')
print(f'[preprocess] 前 20 欄: {list(X_train_tree.columns)[:20]}')

In [ ]:
# 對 Kaggle test.csv 套用同一個 fitted preprocessor (確保 schema 對齊)
test_ids = test_df[ID_COL].values
test_features_raw = test_df.drop(columns=[ID_COL] + ([TARGET] if TARGET in test_df.columns else []),
                                   errors='ignore')
X_test_tree_array = tree_preprocessor.transform(test_features_raw)
X_test_tree = pd.DataFrame(X_test_tree_array, columns=X_train_tree.columns)
print(f'[test] preprocessor.transform 完成: {X_test_tree.shape}')

## Step 4 ─ Target 自動轉換 (log1p) + 餵 daniel pipeline

用我們新加的 `_target_transform`:
- 偵測條件:y_min≥0 + skew≥1.0 + max/p50≥5
- Sticker 全符合 → 自動套 log1p
- 推論期 expm1 + clip≥0 自動還原

In [ ]:
from api.train._target_transform import decide_target_transform, apply_forward, apply_inverse

info = decide_target_transform(y_train.values, 'regression')
print(f'[target_transform] {info}')

y_tr_fit = apply_forward(y_train.values, info).astype(np.float32) if info else y_train.values.astype(np.float32)
X_tr_arr = X_train_tree.values.astype(np.float32)
X_te_arr = X_test_tree.values.astype(np.float32)
print(f'[shape] X_tr={X_tr_arr.shape}  X_te={X_te_arr.shape}  y_tr={y_tr_fit.shape}')
print(f'[y_tr_fit] mean={y_tr_fit.mean():.3f}  std={y_tr_fit.std():.3f}  (log 空間)')

In [ ]:
import pipeline_time as pt
import pipeline as pl

cfg = pt.get_cfg_time(fast=False, n_samples=len(y_tr_fit))
budget = pl.TimeBudget(limit_sec=5400, t_start=time.time())   # 90 分鐘
print(f'[cfg] tabular_trials={cfg.get("tabular_trials")} dl_trials={cfg.get("dl_trials")}')

result = pt.run_regression(
    X_tr_arr, y_tr_fit, X_te_arr,
    cfg, budget,
    skip_tabular=False,
    skip_dl=False,                              # Sticker 樣本多,DL 軌有用
    artifacts_dir='/kaggle/working/artifacts',
    metric='rmse',
)
print(f'\n[done] {len(result.model_tags)} 個 base model 訓練完')
print(f'[models] {result.model_tags}')

## Step 5 ─ OOF 評估 + 每個 base model 排行榜
Metrics 在**原尺度**算才有意義 — 先 inverse 還原。

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true > 0
    if mask.sum() == 0:
        return float('nan')
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / y_true[mask]))

y_tr_orig = y_train.values   # 原尺度真值

# Per-model OOF (還原原尺度後評估)
per_model = []
for tag, oof_pred in zip(result.model_tags, result.all_oof):
    oa = np.asarray(oof_pred).ravel()
    valid = ~np.isnan(oa)
    if valid.sum() < 100:
        continue
    # daniel 內部 y-scaling 已 inverse,我們再 expm1+clip 還原 log1p
    oa_orig = apply_inverse(oa, info) if info else oa
    try:
        m_rmse = float(np.sqrt(mean_squared_error(y_tr_orig[valid], oa_orig[valid])))
        m_r2 = float(r2_score(y_tr_orig[valid], oa_orig[valid]))
        m_mape = mape(y_tr_orig[valid], oa_orig[valid])
        per_model.append((tag, m_r2, m_rmse, m_mape))
    except Exception:
        pass
per_model.sort(key=lambda x: x[1], reverse=True)

print('[Per-Model OOF Leaderboard (原尺度)]')
print(f'  {"tag":<30} {"R²":>8} {"RMSE":>10} {"MAPE":>8}')
for tag, r2, rmse, mp in per_model:
    print(f'  {tag:<30} {r2:>8.4f} {rmse:>10.2f} {mp:>8.3f}')

# Ensemble proxy:all_oof 平均 (跟 individual model 比較哪個強)
valid_oofs = [np.asarray(o).ravel() for o in result.all_oof]
min_len = min(len(o) for o in valid_oofs)
oof_avg = np.mean([o[:min_len] for o in valid_oofs], axis=0)
valid = ~np.isnan(oof_avg)
oof_avg_orig = apply_inverse(oof_avg, info) if info else oof_avg
print(f'\n[Ensemble (avg) OOF]')
print(f'  R²   = {r2_score(y_tr_orig[:min_len][valid], oof_avg_orig[valid]):.4f}')
print(f'  RMSE = {np.sqrt(mean_squared_error(y_tr_orig[:min_len][valid], oof_avg_orig[valid])):.2f}')
print(f'  MAPE = {mape(y_tr_orig[:min_len][valid], oof_avg_orig[valid]):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) Per-model R² bar chart
tags = [t for t, _, _, _ in per_model]
r2s = [r for _, r, _, _ in per_model]
axes[0].barh(tags, r2s, color='steelblue')
axes[0].set_xlabel('OOF R²'); axes[0].set_title('Daniel Pipeline — Per-Model OOF R²')
axes[0].invert_yaxis()

# (2) y_true vs y_pred (OOF ensemble avg)
axes[1].scatter(y_tr_orig[:min_len][valid][:5000], oof_avg_orig[valid][:5000], s=5, alpha=0.3)
lo, hi = 0, max(y_tr_orig.max(), oof_avg_orig.max())
axes[1].plot([lo, hi], [lo, hi], 'r--', alpha=0.5, label='y=x')
axes[1].set_xlabel('True num_sold'); axes[1].set_ylabel('Predicted (OOF ensemble)')
axes[1].set_title('True vs Predicted (前 5000 筆)'); axes[1].legend()

plt.tight_layout(); plt.show()

## Step 6 ─ 生成 submission.csv
test 預測:`result.test_stack` (Meta-learner stacker) → expm1 還原 → clip 非負

In [ ]:
# Stacker test prediction → 原尺度還原
test_pred_log = np.asarray(result.test_stack).ravel()
test_pred = apply_inverse(test_pred_log, info) if info else test_pred_log
test_pred = np.clip(test_pred, 0.0, None)        # 再保險一層 clip≥0

submission = pd.DataFrame({ID_COL: test_ids, TARGET: test_pred})
out_path = '/kaggle/working/submission.csv'
submission.to_csv(out_path, index=False)
print(f'[saved] {out_path}  shape={submission.shape}')
print(f'\n[head]\n{submission.head()}')
print(f'\n[stats]\n{submission[TARGET].describe()}')

## Summary

**Pipeline 全套**:RobustDataCleaner → AutoRouter → PipelineAssembler (tree 軌) → log1p → Daniel pipeline (Tabular + DL) → Nelder-Mead blender + Meta-learner stacker → expm1 + clip≥0 → submission

**輸出**:`/kaggle/working/submission.csv` — 右上 **Submit to Competition**

**想跑更快**:
- `skip_dl=True` 跳過 DL 軌
- `fast=True` 跑精簡版 HPO
- `limit_sec=1800` 把預算砍到 30 分鐘

**想跑更強**:
- `limit_sec` 拉到 9000+(2.5h),讓 Optuna 多搜
- DL 軌維持(本 notebook 已設)
- 加更多 feature(滯後特徵 lag/rolling mean,平台沒自動做但效果顯著)

**Sticker 重點**:metric 是 MAPE,**小銷量段位是 MAPE 殺手**。我們 target log1p 已經幫了大忙(從本機 2.6 拉到接近 gemini 0.3 量級)。